In [ ]:

import sys
import os

# Thêm thư mục gốc của dự án vào sys.path
sys.path.append(os.path.abspath(".."))
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from src.baseline import (
    BASELINE_COLUMNS,
    get_common_eval_timestamps,
    predict_baselines,
)
from src.data import load_hourly_data, load_splits
from src.metrics import (
    calculate_d_train,
    evaluate_forecast,
    evaluate_by_group,
)

# ============================================================
# 0. THIẾT LẬP
# ============================================================

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

CUSTOMER_COL = "MT_158"

os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("models", exist_ok=True)

# ============================================================
# 1. ĐỌC DỮ LIỆU VÀ CÁC TẬP ĐÃ CHIA
# ============================================================

full_df = load_hourly_data(customers=[CUSTOMER_COL])
splits = load_splits(customers=[CUSTOMER_COL])

full_series = full_df[CUSTOMER_COL].astype(float)
train_series = splits["train"][CUSTOMER_COL].astype(float)
val_series = splits["validation"][CUSTOMER_COL].astype(float)
test_series = splits["test"][CUSTOMER_COL].astype(float)

print(f"Chuỗi dữ liệu: {CUSTOMER_COL}")
print(
    f"Train: {len(train_series)} giờ | "
    f"Validation: {len(val_series)} giờ | "
    f"Test: {len(test_series)} giờ"
)

for split_name, split_series in [
    ("Train", train_series),
    ("Validation", val_series),
    ("Test", test_series),
]:
    if not split_series.index.isin(full_series.index).all():
        raise ValueError(
            f"{split_name} có timestamp không tồn tại trong full_series."
        )

# ============================================================
# 2. TÍNH d_train TRÊN TRAIN GỐC
# ============================================================

d_train = calculate_d_train(train_series.to_numpy())

if not np.isfinite(d_train) or d_train <= 0:
    raise ValueError(
        f"d_train không hợp lệ ({d_train}); không thể tính MASE."
    )

print(f"d_train = {d_train:.6f}")

# ============================================================
# 3. FIT SCALER TRÊN TRAIN
# ============================================================
# Chỉ lưu scaler nếu đây là quy ước chung của nhóm.
# Baseline vẫn dự báo trên giá trị điện năng gốc.

scaler = StandardScaler()
scaler.fit(train_series.to_frame())
joblib.dump(scaler, "models/scaler.pkl")

print("Đã lưu models/scaler.pkl")

# ============================================================
# 4. TẠO TIMESTAMP ĐÁNH GIÁ
# ============================================================

val_eval_timestamps = get_common_eval_timestamps(
    full_series, val_series.index
)
test_eval_timestamps = get_common_eval_timestamps(
    full_series, test_series.index
)

if len(val_eval_timestamps) == 0:
    raise ValueError("Không có timestamp hợp lệ trong Validation.")

if len(test_eval_timestamps) == 0:
    raise ValueError("Không có timestamp hợp lệ trong Test.")

print("Số timestamp Validation:", len(val_eval_timestamps))
print("Số timestamp Test:", len(test_eval_timestamps))

# ============================================================
# 5. DỰ BÁO VÀ ĐÁNH GIÁ VALIDATION
# ============================================================

val_predictions = predict_baselines(
    full_series, val_eval_timestamps
)

val_metrics = {}

for col in BASELINE_COLUMNS:
    y_true = val_predictions["y_true"].to_numpy(dtype=float)
    y_pred = val_predictions[col].to_numpy(dtype=float)

    if len(y_true) != len(y_pred):
        raise ValueError(f"Độ dài dữ liệu không khớp ở {col}.")

    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError(f"Có NaN/Inf trong dự báo Validation: {col}.")

    val_metrics[col] = evaluate_forecast(
        y_true, y_pred, d_train=d_train
    )

df_val_results = pd.DataFrame(val_metrics).T
df_val_results.index.name = "model"

if "MAE" not in df_val_results.columns:
    raise KeyError("Không tìm thấy cột MAE trong kết quả đánh giá.")

df_val_results = df_val_results.sort_values("MAE")
best_model = df_val_results.index[0]

print("\n=== KẾT QUẢ VALIDATION ===")
print(df_val_results)
print(f"\nMô hình được chọn theo MAE Validation: {best_model}")

# ============================================================
# 6. DỰ BÁO VÀ ĐÁNH GIÁ TEST
# ============================================================
# Chỉ chọn mô hình dựa trên Validation.
# Test dùng để đánh giá cuối cùng, không dùng để chọn mô hình.

test_predictions = predict_baselines(
    full_series, test_eval_timestamps
)

test_metrics_all = {}

for col in BASELINE_COLUMNS:
    y_true_test_col = test_predictions["y_true"].to_numpy(dtype=float)
    y_pred_test_col = test_predictions[col].to_numpy(dtype=float)

    if not np.isfinite(y_true_test_col).all():
        raise ValueError("Có NaN/Inf trong nhãn Test.")

    if not np.isfinite(y_pred_test_col).all():
        raise ValueError(f"Có NaN/Inf trong dự báo Test: {col}.")

    test_metrics_all[col] = evaluate_forecast(
        y_true_test_col,
        y_pred_test_col,
        d_train=d_train,
    )

df_test_results = pd.DataFrame(test_metrics_all).T
df_test_results.index.name = "model"

print(f"\n=== KẾT QUẢ TEST: {best_model} ===")
print(df_test_results.loc[[best_model]])

# ============================================================
# 7. PHÂN TÍCH SAI SỐ THEO GIỜ VÀ THỨ
# ============================================================

y_true_test = test_predictions["y_true"].to_numpy(dtype=float)
y_pred_best = test_predictions[best_model].to_numpy(dtype=float)
test_ts = pd.DatetimeIndex(
    pd.to_datetime(test_predictions["timestamp"])
)

# Chuẩn bị đúng đầu vào theo API evaluate_by_group trong metrics.py.
df_group_eval = test_predictions.copy()
df_group_eval["timestamp"] = pd.to_datetime(
    df_group_eval["timestamp"]
)

df_error_by_hour = evaluate_by_group(
    df_group_eval,
    pred_col=best_model,
    group_by="hour",
)

df_error_by_dow = evaluate_by_group(
    df_group_eval,
    pred_col=best_model,
    group_by="dayofweek",
)

print("\n=== SAI SỐ THEO GIỜ ===")
print(df_error_by_hour)

print("\n=== SAI SỐ THEO THỨ TRONG TUẦN ===")
print(df_error_by_dow)

# ============================================================
# 8. TOP 10 DỰ BÁO CÓ SAI SỐ LỚN NHẤT
# ============================================================

test_predictions_copy = test_predictions.copy()
test_predictions_copy["abs_error"] = np.abs(
    y_true_test - y_pred_best
)

top10_worst = test_predictions_copy.sort_values(
    "abs_error", ascending=False
).head(10)

top10_worst.to_csv(
    "results/tables/top10_worst_errors.csv",
    index=False,
)

# ============================================================
# 9. BIỂU ĐỒ SO SÁNH BASELINE TRÊN VALIDATION
# ============================================================

ax = df_val_results[["MAE", "RMSE"]].plot(
    kind="bar",
    figsize=(9, 5),
)

ax.set_title("So sánh MAE và RMSE của các Baseline - Validation")
ax.set_xlabel("Mô hình Baseline")
ax.set_ylabel("Sai số")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "results/figures/baseline_val_comparison.png",
    dpi=300,
)
plt.close()

# ============================================================
# 10. BIỂU ĐỒ THỰC TẾ VÀ DỰ BÁO TRÊN TEST
# ============================================================

sample_hours = min(168, len(test_predictions))

plt.figure(figsize=(14, 5))
plt.plot(
    test_ts[:sample_hours],
    y_true_test[:sample_hours],
    label="Thực tế",
    color="black",
    linewidth=1.5,
)
plt.plot(
    test_ts[:sample_hours],
    y_pred_best[:sample_hours],
    label=f"Dự báo: {best_model}",
    color="red",
    linestyle="--",
    linewidth=1.2,
)
plt.title(f"Dự báo một tuần đầu của Test - {best_model}")
plt.xlabel("Thời gian")
plt.ylabel("Phụ tải (kW)")
plt.legend()
plt.tight_layout()

plt.savefig(
    "results/figures/baseline_test_timeseries.png",
    dpi=300,
)
plt.close()

# ============================================================
# 11. BIỂU ĐỒ MAE THEO GIỜ TRONG NGÀY
# ============================================================

plt.figure(figsize=(9, 4))
sns.barplot(
    data=df_error_by_hour,
    x="hour",
    y="MAE",
    color="skyblue",
)
plt.title(f"MAE theo giờ trong ngày - {best_model}")
plt.xlabel("Giờ trong ngày (0–23)")
plt.ylabel("MAE")
plt.xticks(range(0, 24))
plt.tight_layout()

plt.savefig(
    "results/figures/baseline_error_by_hour.png",
    dpi=300,
)
plt.close()

# ============================================================
# 12. XUẤT CÁC FILE KẾT QUẢ
# ============================================================

df_val_results.to_csv(
    "results/tables/baseline_results.csv",
    index=True,
)
df_test_results.to_csv(
    "results/tables/baseline_test_results.csv",
    index=True,
)

val_predictions.to_csv(
    "results/baseline_predictions_validation.csv",
    index=False,
)
test_predictions.to_csv(
    "results/baseline_predictions_test.csv",
    index=False,
)

# File dự báo gộp Validation + Test.
val_output = val_predictions.copy()
val_output.insert(0, "split", "validation")

test_output = test_predictions.copy()
test_output.insert(0, "split", "test")

all_predictions = pd.concat(
    [val_output, test_output],
    ignore_index=True,
)

all_predictions.to_csv(
    "results/baseline_predictions.csv",
    index=False,
)

# Danh sách timestamp đánh giá chung.
eval_timestamps_df = pd.concat(
    [
        pd.DataFrame({
            "split": "validation",
            "timestamp": val_eval_timestamps,
        }),
        pd.DataFrame({
            "split": "test",
            "timestamp": test_eval_timestamps,
        }),
    ],
    ignore_index=True,
)

eval_timestamps_df.to_csv(
    "results/eval_timestamps.csv",
    index=False,
)

df_error_by_hour.to_csv(
    "results/tables/error_by_hour.csv",
    index=False,
)
df_error_by_dow.to_csv(
    "results/tables/error_by_dayofweek.csv",
    index=False,
)

print("\n==============================================")
print("HOÀN THÀNH THÍ NGHIỆM BASELINE")
print("Validation:", "results/tables/baseline_results.csv")
print("Test:", "results/tables/baseline_test_results.csv")
print("Dự báo gộp:", "results/baseline_predictions.csv")
print("Timestamp:", "results/eval_timestamps.csv")
print("Biểu đồ:", "results/figures/")
print("==============================================")


ModuleNotFoundError: No module named 'src'

In [2]:
import sys
!{sys.executable} -m pip install seaborn


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
